# W12 · LLM 심화 통합 실습 — 근거, 구조화, 검증, 비교

오늘 노트북은 **하나의 파일 안에 여러 단계**로 구성됩니다.

1. 기본 예제: naive → grounded → structured → validation
2. 다중 LLM 비교: Gemini / ChatGPT / Claude
3. 추가 self-practice 예제 bank

> 목표: **그럴듯한 답**이 아니라 **근거가 있고, 다시 쓸 수 있고, 다시 확인된 답**을 만드는 것

## 오늘 핵심 개념 4개

### 1) Grounding = 근거 붙이기
- 내가 제공한 문서, 메모, 리뷰를 먼저 주고 답하게 하기

### 2) Structured output = 구조화 출력
- 자유문 대신 JSON/표/체크리스트로 결과 받기

### 3) Validation loop = 검증 루프
- 첫 답을 그대로 믿지 않고 다시 확인하기

### 4) Multi-LLM comparison = 모델 비교
- 같은 프롬프트라도 LLM마다 형식 준수, 과신, 근거 처리 방식이 다를 수 있음

---
## 섹션 1. 기본 실습 데이터 준비

지난주 상권분석 흐름을 이어서, **스터디카페 입지 후보를 판단하는 메모 8개**를 사용합니다.

In [ ]:
import json
import re
import pandas as pd

docs = pd.DataFrame([
    {"doc_id": "D1", "kind": "현장메모", "text": "A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다."},
    {"doc_id": "D2", "kind": "임대료메모", "text": "A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다."},
    {"doc_id": "D3", "kind": "현장메모", "text": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다."},
    {"doc_id": "D4", "kind": "경쟁메모", "text": "B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다."},
    {"doc_id": "D5", "kind": "현장메모", "text": "C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다."},
    {"doc_id": "D6", "kind": "학생설문", "text": "학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다."},
    {"doc_id": "D7", "kind": "운영규정", "text": "밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다."},
    {"doc_id": "D8", "kind": "인터뷰", "text": "시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다."},
])

docs

,doc_id,kind,text
0,D1,현장메모,A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이...
1,D2,임대료메모,A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이...
2,D3,현장메모,B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 ...
3,D4,경쟁메모,"B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카..."
4,D5,현장메모,C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이...
5,D6,학생설문,"학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게..."
6,D7,운영규정,밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지...
7,D8,인터뷰,시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳...


### 참고 자료 및 추가 학습

*   **공식 문서:**
    *   [Colab 문서](https://colab.research.google.com/notebooks/intro.ipynb)
    *   [Google Gemini API 문서](https://ai.google.dev/)

*   **관련 튜토리얼:**
    *   [Python Pandas 시작하기](https://pandas.pydata.org/pandas-docs/stable/getting_started/install.html)
    *   [Matplotlib 튜토리얼](https://matplotlib.org/stable/tutorials/index.html)

*   **유용한 라이브러리:**
    *   `pandas`: 데이터 분석 및 조작
    *   `numpy`: 수치 계산
    *   `matplotlib`, `seaborn`: 데이터 시각화